In [5]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Author: Manuel Elias Orellana Lavayen

Date: 2026-10-05

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.1

debugpy: 1.8.22
numpy  : 2.5.3
pandas : 3.0.5



In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
ruta_base = Path.cwd().parent.parent
ruta_base

WindowsPath('C:/Users/ManuelOrellana/Desktop/Proyectos Aplicaciones/Aplicacion Prediccion Productos/Creacion de Modelos')

In [7]:
def extraer_fechas_festivas (df: pd.DataFrame, num_dias= 5):
    fechas_celebres_local = []
    for localidad in df["locale_name"].unique():
        df_localidad = df[df["locale_name"] == localidad]
        for fecha_celebre in df_localidad["dia-mes"].unique():
            if df_localidad[df_localidad["dia-mes"] == fecha_celebre].shape[0] >= num_dias:
                fechas_celebres_local.append(fecha_celebre)
    return fechas_celebres_local

In [8]:
df = pd.read_csv(filepath_or_buffer= ruta_base/"Analisis y entrenamiento de modelos/Datasets/Modificados/holidays_local.csv", parse_dates= ["date"])
df["dia-mes"] = df["date"].dt.strftime("%m-%d")
df.head()

,date,locale_name,es_feriado_local,dia-mes
0,2012-03-02,Manta,True,03-02
1,2012-04-12,Cuenca,True,04-12
2,2012-04-14,Libertad,True,04-14
3,2012-04-21,Riobamba,True,04-21
4,2012-05-12,Puyo,True,05-12


In [9]:
df = df.drop(columns= ["date", "es_feriado_local"])
df = df[df["dia-mes"].isin(extraer_fechas_festivas(df= df))].drop_duplicates()
df = df.rename(columns= {"locale_name": "ciudad"})
df.to_csv(ruta_base/"CreacionBaseConocimiento/BaseConocimiento/holidays_local.csv", index= None)

In [10]:
df = pd.read_csv(filepath_or_buffer= ruta_base/"Analisis y entrenamiento de modelos/Datasets/Modificados/holidays_regional.csv", parse_dates= ["date"])
df["dia-mes"] = df["date"].dt.strftime("%m-%d")
df.head()

,date,locale_name,es_feriado_regional,dia-mes
0,2012-04-01,Cotopaxi,True,04-01
1,2012-06-25,Imbabura,True,06-25
2,2012-11-06,Santo Domingo de los Tsachilas,True,11-06
3,2012-11-07,Santa Elena,True,11-07
4,2013-04-01,Cotopaxi,True,04-01


In [11]:
df = df.drop(columns= ["date", "es_feriado_regional"])
df = df[df["dia-mes"].isin(extraer_fechas_festivas(df= df))].drop_duplicates()
df = df.rename(columns= {"locale_name": "provincia_estado"})
df.to_csv(ruta_base/"CreacionBaseConocimiento/BaseConocimiento/holidays_regional.csv", index= None)

In [12]:
df = pd.read_csv(filepath_or_buffer= ruta_base/"Analisis y entrenamiento de modelos/Datasets/Modificados/holidays_nacional.csv", parse_dates= ["date"])
df["dia-mes"] = df["date"].dt.strftime("%m-%d")
df.head()

,date,es_feriado_nacional,es_evento,dia-mes
0,2012-08-10,True,False,08-10
1,2012-10-09,False,False,10-09
2,2012-10-12,True,False,10-12
3,2012-11-02,True,False,11-02
4,2012-11-03,True,False,11-03


In [13]:
fechas_celebres_nacional = []
for fecha_celebre in df["dia-mes"].unique():
    if df[df["dia-mes"] == fecha_celebre].shape[0] >= 5:
        fechas_celebres_nacional.append(fecha_celebre)

df = df.drop(columns= ["date", "es_feriado_nacional", "es_evento"])
df = df[df["dia-mes"].isin(fechas_celebres_nacional)].drop_duplicates()
df.to_csv(ruta_base/"CreacionBaseConocimiento/BaseConocimiento/holidays_nacional.csv", index= None)

In [14]:
df = pd.read_csv(filepath_or_buffer= ruta_base/"Analisis y entrenamiento de modelos/Datasets/Originales/holidays_events.csv", parse_dates= ["date"])
df[df["type"] == "Event"]["description"].unique()

<ArrowStringArray>
[                                'Dia de la Madre',
           'Inauguracion Mundial de futbol Brasil',
         'Mundial de futbol Brasil: Ecuador-Suiza',
      'Mundial de futbol Brasil: Ecuador-Honduras',
       'Mundial de futbol Brasil: Ecuador-Francia',
      'Mundial de futbol Brasil: Octavos de Final',
      'Mundial de futbol Brasil: Cuartos de Final',
           'Mundial de futbol Brasil: Semifinales',
 'Mundial de futbol Brasil: Tercer y cuarto lugar',
                 'Mundial de futbol Brasil: Final',
                                    'Black Friday',
                                    'Cyber Monday',
                                'Terremoto Manabi',
                              'Terremoto Manabi+1',
                              'Terremoto Manabi+2',
                              'Terremoto Manabi+3',
                              'Terremoto Manabi+4',
                              'Terremoto Manabi+5',
                              'Terremoto Mana

In [15]:
serie_tiempo_eventos = pd.date_range(start= "2016-01-01", end= "2018-12-31" )
serie_tiempo_eventos = pd.DataFrame({"fecha": serie_tiempo_eventos})
serie_tiempo_eventos["año"] = serie_tiempo_eventos['fecha'].dt.year
serie_tiempo_eventos["mes"] = serie_tiempo_eventos['fecha'].dt.month
serie_tiempo_eventos["semana_del_mes"] = np.ceil(serie_tiempo_eventos['fecha'].dt.day / 7).astype(int)
serie_tiempo_eventos["num_dia_semana"] = serie_tiempo_eventos['fecha'].dt.dayofweek
serie_tiempo_eventos["dia"] = serie_tiempo_eventos['fecha'].dt.day
serie_tiempo_eventos["tipo_evento"] = None

In [16]:
# Dia se la madre
serie_tiempo_eventos.loc[
    (serie_tiempo_eventos["mes"] == 5) &
    (serie_tiempo_eventos["semana_del_mes"] == 2) &
    (serie_tiempo_eventos["num_dia_semana"] == 6), "tipo_evento"] = "Dia de la Madre"

# Black Friday
serie_tiempo_eventos.loc[
    (serie_tiempo_eventos["mes"] == 11) &
    (serie_tiempo_eventos["num_dia_semana"] == 4) &
    (serie_tiempo_eventos["dia"] >= 23) &
    (serie_tiempo_eventos["dia"] <= 29), "tipo_evento"] = "Black Friday"

# Cyber Monday (caso noviembre)
serie_tiempo_eventos.loc[
    (serie_tiempo_eventos["mes"] == 11) &
    (serie_tiempo_eventos["semana_del_mes"] == 5) &
    (serie_tiempo_eventos["num_dia_semana"] == 0), "tipo_evento"] = "Cyber Monday"

# Cyber Monday (caso diciembre)
serie_tiempo_eventos.loc[
    (serie_tiempo_eventos["mes"] == 12) &
    (serie_tiempo_eventos["semana_del_mes"] == 1) &
    (serie_tiempo_eventos["num_dia_semana"] == 0), "tipo_evento"] = "Cyber Monday"

serie_tiempo_eventos= serie_tiempo_eventos.dropna()
serie_tiempo_eventos = serie_tiempo_eventos.drop(columns= ["año", "mes", "semana_del_mes", "num_dia_semana", "dia"])

In [17]:
serie_tiempo_eventos.to_csv(ruta_base/"CreacionBaseConocimiento/BaseConocimiento/eventos.csv", index= None)